
# 01_18 — Avaliação mínima de exequibilidade de um projeto ABM/MBA com apoio de LLM

Este notebook ajuda você a fazer uma **primeira avaliação de viabilidade** de uma ideia de projeto de simulação baseada em agentes, também chamada aqui de **MBA/ABM**.

A pergunta central é:

> Minha ideia pode virar um primeiro protótipo simples, executável no JupyterLab do SimServ?

Você **não precisa saber responder tudo**. A proposta deste notebook é justamente transformar uma ideia inicial, ainda incompleta, em uma avaliação preliminar mais organizada.

---

## Como usar

1. Leia as instruções.
2. Preencha o formulário mínimo com o que souber.
3. Rode as células do notebook em ordem.
4. Gere um prompt estruturado para o LLM.
5. Se a IA local do SimServ estiver disponível, rode a consulta automaticamente.
6. Salve a resposta como relatório na sua área de trabalho.

> Recomenda-se abrir este notebook em uma pasta de orientação, como `/sobre_ambientes_simserv`, e salvar uma cópia em sua pasta de trabalho, como `/home/jovyan/work`, antes de editar.



## 1. O que significa “exequibilidade mínima”?

Neste notebook, **exequibilidade mínima** quer dizer:

- a ideia é clara o suficiente para gerar um primeiro modelo simplificado;
- o número inicial de agentes pode ser pequeno;
- o número de passos de simulação pode ser limitado;
- as regras podem ser reduzidas a uma primeira versão;
- os resultados podem ser observados com gráficos e tabelas simples;
- o código pode ser testado em partes;
- o notebook não deve depender de uma simulação grande logo no início.

A avaliação não é uma garantia de sucesso. Ela é uma forma de evitar começar por um projeto grande demais, ambíguo demais ou computacionalmente pesado demais.



## 2. Critério prático

Para o primeiro protótipo, prefira algo como:

| Item | Recomendação inicial |
|---|---:|
| Tipos de agentes | 1 a 3 tipos |
| Número de agentes | começar com 10, 50 ou 100 |
| Passos de simulação | começar com 20, 50 ou 100 |
| Repetições independentes | começar com 1 a 5 |
| Gráficos | poucos e simples |
| Salvamento de dados | estatísticas agregadas primeiro |
| Interface gráfica | deixar para uma versão posterior |

Uma regra de ouro:

> Comece pequeno, meça o tempo, aumente aos poucos.

Se o modelo precisar que todos os agentes comparem suas decisões com todos os outros agentes, o custo pode crescer muito rapidamente. Nesse caso, o LLM deve ajudar a sugerir simplificações.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# 3. Verificação simples do ambiente e da pasta de trabalho

from pathlib import Path
import os
import sys
import platform
import datetime

HOME = Path.home()
CANDIDATAS_TRABALHO = [
    Path.cwd().parent.parent / "work",
    HOME / "work",
    HOME / "laboratorio",
    Path("/home/jovyan/work"),
    Path("/home/jovyan/laboratorio"),
]

PASTA_TRABALHO = None
for pasta in CANDIDATAS_TRABALHO:
    if pasta.exists() and os.access(pasta, os.W_OK):
        PASTA_TRABALHO = pasta
        break

if PASTA_TRABALHO is None:
    PASTA_TRABALHO = Path.cwd()

PASTA_SAIDA = PASTA_TRABALHO / "avaliacoes_exequibilidade"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Pasta atual:", Path.cwd())
print("Pasta de trabalho escolhida:", PASTA_TRABALHO)
print("Pasta de saída:", PASTA_SAIDA)
print("Pode escrever na pasta de saída?", os.access(PASTA_SAIDA, os.W_OK))



## 4. Formulário mínimo do projeto

Preencha apenas o que você souber. Pode deixar campos incompletos.

O objetivo é permitir que o LLM faça uma **avaliação inicial**, com hipóteses explícitas, sem exigir que você já tenha todos os detalhes técnicos.


In [ ]:
# 5. Preencha este formulário mínimo
# Substitua os textos entre aspas pelo conteúdo do seu projeto.
# Pode deixar "não sei ainda" quando necessário.

nome_do_aluno = "nome_do_aluno"

projeto = {
    "titulo": "Meu projeto de simulação baseada em agentes",
    
    "area_ou_tema": "Exemplo: mobilidade urbana, adoção de tecnologia, epidemias, mercado, ecologia, educação, opinião pública etc.",
    
    "ideia_em_poucas_linhas": "Descreva a ideia principal do projeto em 3 a 10 linhas.",
    
    "pergunta_principal": "Qual pergunta você gostaria que a simulação ajudasse a investigar?",
    
    "agentes_imaginados": "Quem são os agentes? Exemplo: pessoas, empresas, veículos, escolas, partículas, consumidores etc.",
    
    "tipos_de_agentes": "Há tipos diferentes de agentes? Exemplo: consumidores R e S, motoristas e pedestres, alunos e professores etc. Se não souber, escreva 'não sei ainda'.",
    
    "ambiente": "O modelo precisa de espaço físico, rede, grade, grupos, cidade, sala, mercado, internet, ou pode ser uma população sem espaço explícito?",
    
    "regras_de_comportamento": "Que decisões os agentes tomam? O que muda em cada passo da simulação?",
    
    "interacoes": "Os agentes interagem com vizinhos, com todos, com grupos, com o ambiente, ou não interagem diretamente?",
    
    "parametros": "Que parâmetros você imagina variar? Exemplo: número de agentes, probabilidade de adoção, taxa de contato, intensidade de preferência etc.",
    
    "resultados_esperados": "Que saídas você gostaria de ver? Exemplo: gráficos, tabelas, distribuição final, evolução temporal etc.",
    
    "tamanho_desejado": "Você imaginou quantos agentes, passos e repetições? Se não souber, escreva 'não sei ainda'.",
    
    "nivel_de_experiencia": "iniciante",
    
    "preocupacoes": "O que parece difícil no projeto? Exemplo: gerar código, tempo de processamento, muitas regras, muitos agentes, gráficos etc."
}

for chave, valor in projeto.items():
    print(f"{chave}: {valor}\n")


In [ ]:
# 6. Coleta de informações úteis do ambiente
# Essas informações serão incluídas no prompt para o LLM.

import importlib

bibliotecas = ["numpy", "pandas", "matplotlib", "networkx", "mesa", "requests"]
info_bibliotecas = []

for nome in bibliotecas:
    try:
        modulo = importlib.import_module(nome)
        versao = getattr(modulo, "__version__", "versão não informada")
        info_bibliotecas.append(f"{nome}: disponível ({versao})")
    except Exception:
        info_bibliotecas.append(f"{nome}: não disponível")

try:
    import psutil
    memoria_gb = round(psutil.virtual_memory().total / (1024**3), 2)
    cpu_count = psutil.cpu_count(logical=True)
except Exception:
    memoria_gb = "não detectada"
    cpu_count = os.cpu_count()

ambiente_simserv = {
    "python": sys.version.split()[0],
    "sistema": platform.platform(),
    "cpu_logicos": cpu_count,
    "memoria_total_gb": memoria_gb,
    "pasta_atual": str(Path.cwd()),
    "pasta_trabalho": str(PASTA_TRABALHO),
    "pasta_saida": str(PASTA_SAIDA),
    "bibliotecas": info_bibliotecas,
    "observacao": "Ambiente educacional em JupyterLab. Preferir protótipos pequenos, código simples, gráficos simples e salvamento de resultados em arquivos."
}

ambiente_simserv



## 7. Prompt de avaliação mínima

A próxima célula cria um prompt que pede ao LLM uma avaliação inicial de exequibilidade.

A recomendação central incluída no prompt é:

> Se o aluno não forneceu todas as informações, não interrompa a avaliação. Faça hipóteses explícitas, proponha uma versão mínima viável e liste no máximo cinco perguntas essenciais para melhorar a análise.


In [ ]:
# 8. Construção do prompt para o LLM

import json
import textwrap
import re


def criar_slug(texto, limite=60):
    texto = texto.lower().strip()
    texto = re.sub(r"[^a-z0-9áàãâéêíóôõúç_-]+", "_", texto)
    texto = texto.strip("_")
    if not texto:
        texto = "projeto"
    return texto[:limite]


def montar_prompt_exequibilidade(projeto, ambiente):
    projeto_json = json.dumps(projeto, ensure_ascii=False, indent=2)
    ambiente_json = json.dumps(ambiente, ensure_ascii=False, indent=2)
    
    prompt = f"""
Você é um assistente didático para estudantes iniciantes que estão usando o SimServ em JupyterLab para criar simulações baseadas em agentes, também chamadas de MBA ou ABM.

Tarefa:
Avalie a EXEQUIBILIDADE MÍNIMA do projeto descrito pelo aluno. A avaliação deve servir para decidir se a ideia pode virar um primeiro protótipo simples em notebook JupyterLab.

Instrução muito importante:
O aluno provavelmente não sabe responder todas as perguntas técnicas. Portanto, NÃO exija uma especificação completa. Se faltarem informações, continue mesmo assim, faça hipóteses explícitas e indique no máximo 5 perguntas essenciais para melhorar a proposta.

Objetivo da resposta:
Ajudar o aluno a começar pequeno, com um primeiro protótipo executável, evitando travar o notebook com muitos agentes, muitas regras, muitos dados ou interação todos-com-todos.

Descrição do projeto fornecida pelo aluno:
{projeto_json}

Informações do ambiente detectadas pelo notebook:
{ambiente_json}

Responda em português, com linguagem clara para iniciante, usando a estrutura abaixo:

1. Diagnóstico geral em poucas linhas
2. Classificação inicial: Verde, Amarelo ou Vermelho
   - Verde: viável como primeiro protótipo simples
   - Amarelo: viável com simplificações
   - Vermelho: grande ou ambíguo demais para começar sem reformulação
3. Hipóteses usadas por você porque a descrição ainda está incompleta
4. Versão mínima viável do projeto
5. Agentes mínimos necessários
6. Ambiente mínimo necessário
7. Regras mínimas para a primeira versão
8. Parâmetros iniciais seguros
   - número inicial de agentes
   - número inicial de passos
   - número inicial de repetições
9. Estimativa qualitativa de custo computacional
   - diga se parece aproximadamente O(N), O(N log N), O(N²) ou outro
   - explique em linguagem simples o que pode deixar o modelo lento
10. Riscos técnicos principais
11. Qualidade esperada da geração de código por LLM
   - a descrição atual é suficiente para gerar código?
   - o código deve começar em Python simples, Mesa, NetworkX, pandas, NumPy ou outra opção?
   - que pedido o aluno deve fazer primeiro ao LLM?
12. Dados e gráficos mínimos que devem ser salvos
13. Primeiro experimento recomendado para teste
14. No máximo 5 perguntas essenciais que o aluno deveria responder depois
15. Próximo passo prático

Regras para sua resposta:
- Não invente limites numéricos absolutos do SimServ.
- Não prometa desempenho.
- Recomende começar pequeno e medir tempo de execução.
- Dê preferência a código simples e testável.
- Se houver interação todos-com-todos, alerte sobre custo O(N²).
- Se a ideia estiver muito grande, proponha uma versão reduzida, não apenas recuse.
"""
    return textwrap.dedent(prompt).strip()

prompt_exequibilidade = montar_prompt_exequibilidade(projeto, ambiente_simserv)

slug = criar_slug(projeto.get("titulo", "projeto"))
data_hora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
arquivo_prompt = PASTA_SAIDA / f"prompt_exequibilidade_{slug}_{data_hora}.txt"
arquivo_prompt.write_text(prompt_exequibilidade, encoding="utf-8")

print("Prompt criado e salvo em:")
print(arquivo_prompt)
print("\n--- INÍCIO DO PROMPT ---\n")
print(prompt_exequibilidade[:4000])
if len(prompt_exequibilidade) > 4000:
    print("\n[Prompt truncado na tela. O arquivo contém o prompt completo.] ")
print("\n--- FIM DA PRÉVIA ---")



## 8. Consulta opcional à IA local do SimServ

A próxima célula tenta encontrar um servidor Ollama disponível. Em algumas instalações do SimServ, a IA local pode estar acessível como:

- `http://ollama:11434`
- `http://localhost:11434`
- `http://127.0.0.1:11434`

Se não funcionar, copie o prompt salvo em arquivo e use outro meio indicado pelo professor ou monitor.


In [ ]:
# 9. Conexão com LLM via Ollama Cloud
# Se OLLAMA_API_KEY estiver definida no .env, tenta Ollama Cloud.
# Caso contrário, tenta servidor Ollama local.

import os
import requests

base_url_ollama = os.getenv('OLLAMA_BASE_URL', 'https://api.ollama.com').rstrip('/')
api_key = os.getenv('OLLAMA_API_KEY')
modelo_escolhido = os.getenv('OLLAMA_MODEL', 'ministral-3:3b')

headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
if api_key:
    headers['Authorization'] = f'Bearer {api_key}'

urls = [base_url_ollama]
for fb in ['http://ollama:11434', 'http://localhost:11434', 'http://127.0.0.1:11434']:
    if fb not in urls:
        urls.append(fb)

endpoint_encontrado = None
for url in urls:
    try:
        r = requests.get(f'{url}/api/tags', headers=headers, timeout=15)
        if r.ok:
            try:
                data = r.json()
                if 'models' in data:
                    endpoint_encontrado = url
                    break
            except Exception:
                pass
    except Exception:
        pass

base_url_ollama = endpoint_encontrado

print("Ollama encontrado em:", base_url_ollama)
print("Modelo configurado:", modelo_escolhido)


In [ ]:
# 10. Rodar a consulta ao LLM
# Se falhar, use o arquivo de prompt salvo.

resposta_llm = None

if base_url_ollama is None:
    print("LLM não disponível (nenhum endpoint Ollama encontrado).")
    print("Use o arquivo de prompt salvo em:")
    print(arquivo_prompt)
else:
    print(f"Consultando modelo {modelo_escolhido} em {base_url_ollama}...")
    try:
        payload = {
            'model': modelo_escolhido,
            'messages': [{'role': 'user', 'content': prompt_exequibilidade}],
            'stream': False,
            'options': {'temperature': 0.2},
        }
        resp = requests.post(
            f'{base_url_ollama}/api/chat',
            headers=headers,
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        resposta_llm = resp.json().get('message', {}).get('content', '').strip()
        if not resposta_llm:
            resposta_llm = resp.json().get('response', '').strip()
        print("Resposta recebida. Primeira prévia:\n")
        print(resposta_llm[:4000])
        if len(resposta_llm) > 4000:
            print("\n[Resposta truncada na tela. A célula seguinte salva o relatório completo.]")
    except Exception as exc:
        print(f"Erro ao consultar LLM em {base_url_ollama}: {exc}")
        print("Prompt salvo em:", arquivo_prompt)


In [ ]:
# 11. Salvar relatório da avaliação

from IPython.display import Markdown, display

if resposta_llm is None:
    print("Nenhuma resposta do LLM foi gerada nesta sessão.")
    print("O prompt está salvo em:")
    print(arquivo_prompt)
else:
    arquivo_relatorio_md = PASTA_SAIDA / f"relatorio_exequibilidade_{slug}_{data_hora}.md"
    conteudo = f"""# Relatório de exequibilidade mínima

Aluno: {nome_do_aluno}

Projeto: {projeto.get('titulo', '')}

Data/hora: {datetime.datetime.now()}

Modelo LLM: {modelo_escolhido}

Base URL Ollama: {base_url_ollama}

---

{resposta_llm}
"""
    arquivo_relatorio_md.write_text(conteudo, encoding="utf-8")
    print("Relatório salvo em:")
    print(arquivo_relatorio_md)
    display(Markdown(resposta_llm))



## 9. Como interpretar a resposta

Use a classificação do LLM como **orientação inicial**, não como decisão definitiva.

### Verde

O projeto parece adequado para um primeiro protótipo simples. Ainda assim, comece pequeno.

### Amarelo

O projeto é interessante, mas precisa ser reduzido. Use a versão mínima sugerida antes de tentar o modelo completo.

### Vermelho

A ideia pode ser boa, mas está grande, vaga ou pesada demais para começar. Reformule a pergunta, reduza agentes, reduza regras, reduza interações e tente novamente.

---

## Pergunta final para discussão com monitor ou professor

Depois de obter a resposta do LLM, tente responder:

> Qual é o menor experimento que eu consigo rodar para testar se minha ideia começa a funcionar?



## 10. Prompt curto alternativo

Se você quiser uma avaliação ainda mais simples, pode copiar este modelo:

```text
Quero fazer uma avaliação mínima de exequibilidade de um projeto de simulação baseada em agentes no JupyterLab do SimServ.

Minha ideia é:
[descreva em poucas linhas]

Ainda não sei todos os detalhes técnicos. Por favor:
1. diga se a ideia parece viável como primeiro protótipo;
2. proponha uma versão mínima;
3. sugira número inicial de agentes, passos e repetições;
4. indique o que pode ficar pesado;
5. diga se a descrição já permite gerar código simples;
6. liste no máximo 5 perguntas essenciais que eu devo responder depois;
7. classifique como Verde, Amarelo ou Vermelho.

Não invente limites absolutos do computador. Recomende começar pequeno e medir o tempo de execução.
```
